# run

The hourly run of this workspace, what `process_data.yml` does on GitHub: dbt-duckdb downloads
the AEMO files into `nem/Files/csv` and writes the `landing` and `mart` tables to the `nem`
lakehouse through the OneLake Iceberg REST catalog. `run_pipeline` calls it on a schedule.

The dbt project is the folder `scripts/deploy_fabric.py` uploaded to `nem/Files/project`
(its `COMMIT` file names the commit). dbt is not preinstalled in a Fabric notebook: it is
pip-installed from the project's `requirements.txt`, the pins the GitHub runs use.


In [ ]:
# Files fetched per feed per run, and files turned into fact rows per feed per run: the
# defaults of process_data.yml.
download_limit = "200"
process_limit = "300"


In [ ]:
# 1. SETTINGS: the environment process_data.yml gives dbt (AGENTS.md, "Env contract"), from
# this workspace: its id, the `nem` lakehouse's id, and a OneLake token of whoever runs the
# notebook.
import json
import os
import shlex
import shutil
import subprocess
import sys
import urllib.request

import notebookutils

workspace = notebookutils.runtime.context["currentWorkspaceId"]


def fabric(path):
    """One GET of the Fabric REST API for this workspace."""
    request = urllib.request.Request(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace}{path}",
        headers={"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"})
    with urllib.request.urlopen(request) as r:
        return json.loads(r.read())


found = [i["id"] for i in fabric("/lakehouses")["value"] if i["displayName"] == "nem"]
if not found:
    raise RuntimeError("no lakehouse `nem` in this workspace: deploy with deploy_fabric.yml")
lakehouse = found[0]

# The runners need curl and a CA file: the azure extension's default transport fails the
# OneLake TLS handshake.
ca = "/etc/ssl/certs/ca-certificates.crt"
if not os.path.exists(ca):
    import certifi
    ca = certifi.where()
os.environ.update({
    "ONELAKE_ENDPOINT": "https://onelake.table.fabric.microsoft.com/iceberg",
    "ONELAKE_TOKEN": notebookutils.credentials.getToken("storage"),
    "WAREHOUSE_PATH": f"{workspace}/{lakehouse}",
    "FILES_PATH": f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{lakehouse}/Files",
    "AZURE_TRANSPORT_OPTION_TYPE": "curl",
    "CURL_CA_INFO": ca,
    "download_limit": str(download_limit),
    "process_limit": str(process_limit),
    # So dbt's log reaches the cells below line by line, not in blocks.
    "PYTHONUNBUFFERED": "1",
})
print(f"WAREHOUSE_PATH={os.environ['WAREHOUSE_PATH']}")


In [ ]:
# 2. THE PROJECT: the folder the deploy uploaded, copied to local disk, where dbt writes its
# target/ and logs/.
PROJECT = "/lakehouse/default/Files/project"
if not os.path.exists(f"{PROJECT}/COMMIT"):
    raise RuntimeError(f"no dbt project in {PROJECT}: deploy with deploy_fabric.yml")
work = "/home/trusted-service-user/work" if os.path.isdir("/home/trusted-service-user/work") else "/tmp"
repo = os.path.join(work, "analytics-as-code")
shutil.rmtree(repo, ignore_errors=True)
shutil.copytree(PROJECT, repo)
print(f"{repo} @ {open(os.path.join(repo, 'COMMIT')).read().strip()}")


In [ ]:
# 3. INSTALL dbt, dbt-duckdb and duckdb, pinned in requirements.txt. dbt runs as a
# subprocess, so this kernel never imports the duckdb it replaces.
def sh(*cmd):
    """One command in the project folder, its log printed here line by line. A failure
    raises, so a scheduled run goes red."""
    p = subprocess.Popen(list(cmd), cwd=repo, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"{shlex.join(cmd)} failed (exit {p.returncode}); see the log above")


sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")


In [ ]:
# 4. RUN: the two steps of process_data.yml, the landing facts and then the tables the
# semantic model reads, which are built from them.
def dbt(*args):
    # `python -m`, not the `dbt` executable: pip may put that outside the notebook's PATH.
    sh(sys.executable, "-m", "dbt.cli.main", "run", "--target", "prod", "--profiles-dir", ".", *args)


dbt("--exclude", "tag:powerbi")
dbt("--select", "tag:powerbi")
